# BreakHis EDA

1. Parse the filenames into a tidy DataFrame.
2. Verify that the patient-level split has no leakage.
3. Plot class & magnification distributions.
4. Visualize 5 images per (subtype × magnification) cell.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from src.dataset import (SUBTYPE_CODES, SUBTYPE_ORDER, VALID_MAGNIFICATIONS,
                         patient_level_split, scan_directory, split_summary,
                         to_dataframe)

DATA_ROOT = '../data/BreaKHis_v1'  # adjust to your local path
df = to_dataframe(scan_directory(DATA_ROOT))
print(f'{len(df):,} images   {df.patient_id.nunique()} patients')
df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['binary'].value_counts().plot.bar(ax=axes[0], title='Benign vs Malignant (images)')
df['subtype'].value_counts().reindex(SUBTYPE_ORDER).plot.bar(
    ax=axes[1], title='Subtype distribution (images)', color='tab:orange')
df['magnification'].value_counts().sort_index().plot.bar(
    ax=axes[2], title='Magnification distribution', color='tab:green')
plt.tight_layout(); plt.show()

In [ ]:
# Patient-level split & leakage check
splits = patient_level_split(df, val_size=0.15, test_size=0.15, random_state=42)
print(split_summary(splits).to_string(index=False))

train_p = set(splits['train'].patient_id)
val_p   = set(splits['val'].patient_id)
test_p  = set(splits['test'].patient_id)
assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p)
print('OK -- no patient appears in two splits.')
for name, sub in splits.items():
    n_mags = sub.groupby('patient_id').magnification.nunique().value_counts().to_dict()
    print(f'  {name}: patients-by-#mags = {n_mags}')

In [ ]:
# 5 images per subtype per magnification
rng = np.random.default_rng(0)
for sub in SUBTYPE_ORDER:
    fig, axes = plt.subplots(len(VALID_MAGNIFICATIONS), 5,
                             figsize=(15, 3 * len(VALID_MAGNIFICATIONS)))
    for i, mag in enumerate(VALID_MAGNIFICATIONS):
        pool = df[(df.subtype == sub) & (df.magnification == mag)]
        if pool.empty:
            for ax in axes[i]: ax.axis('off')
            continue
        picks = pool.sample(n=min(5, len(pool)), random_state=int(rng.integers(0, 10**6)))
        for j, (_, row) in enumerate(picks.iterrows()):
            img = Image.open(row.path)
            axes[i, j].imshow(img); axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_ylabel(f'{mag}X', rotation=0, labelpad=30)
    fig.suptitle(f'{sub} - {SUBTYPE_CODES[sub]}', fontsize=14)
    plt.tight_layout(); plt.show()